In [1]:
from pathlib import Path

home = Path.home()
shared = Path('/home/shared/owner')
cop = shared/'CoP_files'
root1 = cop/'input/CF6_IORSIM_AH/CF6_IORSIM'
root2 = cop/'input/IORSIM_BASECASE_SURF_RES/IORSIM_BASECASE_INPUT'
root3 = cop/'output/K-9A_PILOT/K-9A_PILOT'
gsg1 = root1.parent/'CF6_IORSIM_POROSITY.GSG'
gsg2 = root1.parent/'CF6_IORSIM_CONNATE_WATER_SATURATION_KR.GSG'
gsg3 = root1.parent/'CF6_IORSIM.GSG'
gsg4 = root2.parent/'IORSIM_BASECASE_INPUT_CONNATE_WATER_SATURATION.GSG'
gsg5 = root2.parent/'IORSIM_BASECASE_INPUT_POROSITY.GSG'
gsg6 = root2.parent/'IORSIM_BASECASE_INPUT.GSG'
gsg7 = root3.parent/'K-9A_PILOT.GSG'
gsg8 = root3.parent/'K-9A_PILOT_POROSITY.GSG'


In [ ]:
from pathlib import Path
from IORlib.GSG import Data

home = Path.home()
shared = Path('/home/shared/owner')
cop = shared/'CoP_files'
root1 = cop/'input/CF6_IORSIM_AH/CF6_IORSIM'
root2 = cop/'output/K-9A_PILOT/K-9A_PILOT'
reg = '_REGIONS.GSG'
ntg = '_NET_TO_GROSS_RATIO.GSG'
perm = '_PERM_I.GSG'
#gsg1 = str(root1) + 
#gsg2 = str(root2) + '_REGIONS.GSG'
#gsg3 = str(root1) + 
#gsg4 = str(root1) + '_PERM_I.GSG'

dim = (40, 1, 1)
rundir = root1.parent
backup = rundir/'GSG_backup'
backup.mkdir(exist_ok=True)
for gsg in rundir.glob('*.GSG'):
    print()
    print(gsg)
    data = Data().from_file(gsg)
    backup_path = backup/gsg.name
    print(f'Moving {gsg} -> {backup_path}')
    gsg.rename(backup_path)
    if data.is_grid():
        data.new_grid(dim)
    else:
        data.set_values(dim)
    data.to_file(gsg)

    print(data)
    testfile = 'TEST.GSG'
    data.to_file(testfile)
    test = Data().from_file(testfile)
    print(testfile)
    print(test)




/home/shared/owner/CoP_files/input/CF6_IORSIM_AH/CF6_IORSIM_PRESSURE.GSG
PROP       : [0, 1]
PRS        : [b'ca  ', 1, 20][300.0, 300.0, 300.0] ... [300.0, 300.0, 300.0]
CASE_PROPS : [0, b's   ', 0, 1, b'ca  s   ', 1]
PRESSURE   : [b's   ', 1, 1]
TEST.GSG
PROP       : [0, 1]
PRS        : [b'ca  ', 1, 20][300.0, 300.0, 300.0] ... [300.0, 300.0, 300.0]
CASE_PROPS : [0, b's   ', 0, 1, b'ca  s   ', 1]
PRESSURE   : [b's   ', 1, 1]

/home/shared/owner/CoP_files/input/CF6_IORSIM_AH/CF6_IORSIM_PERM_K.GSG
PROP       : [0, 1]
PERM       : [b'ca  ', 1, 20][3.809999942779541, 3.809999942779541, 3.809999942779541] ... [3.809999942779541, 3.809999942779541, 3.809999942779541]
CASE_PROPS : [0, b's   ', 0, 1, b'ca  s   ', 1]
PERM_K     : [b'g   ', 1, 1]
TEST.GSG
PROP       : [0, 1]
PERM       : [b'ca  ', 1, 20][3.809999942779541, 3.809999942779541, 3.809999942779541] ... [3.809999942779541, 3.809999942779541, 3.809999942779541]
CASE_PROPS : [0, b's   ', 0, 1, b'ca  s   ', 1]
PERM_K     : [b'g   ', 1,

In [26]:
d7 = Data().from_file(gsg7)
d7['AREAL'][1][150:160]

[[150, 4, 302, 150, 151, 303],
 [151, 4, 304, 152, 153, 305],
 [152, 4, 305, 153, 154, 306],
 [153, 4, 306, 154, 155, 307],
 [154, 4, 307, 155, 156, 308],
 [155, 4, 308, 156, 157, 309],
 [156, 4, 309, 157, 158, 310],
 [157, 4, 310, 158, 159, 311],
 [158, 4, 311, 159, 160, 312],
 [159, 4, 312, 160, 161, 313]]

In [ ]:

def set_areal(dim):
    areal = []
    n = 0
    for j in range(dim[1]):
        a = j*(dim[0]+1)
        b = a + (dim[0]+1)
        for i in range(dim[0]):
            areal.append((n, 4, b+i,  a+i , a+i+1, b+i+1))
            n += 1
    return areal



222 ms ± 347 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [7]:
A = [[1,2,3], [4,5]]
a, b = A
b

[4, 5]

In [ ]:
import os
import struct

from anyio import key

class GsgData:
    def __init__(self):
        self.int_data = []
        self.float_data = []
        self.double_data = []

class GsgOutput:
    def __init__(self):
        self.headers = []
        self.data_vector = []

def read_keyword(file):
    # Read the size of the header
    size_header = struct.unpack('<i', file.read(4))[0]

    # Read the keyword
    keyword = file.read(size_header).decode('utf-8')
    print('kw', keyword)
    keyword = keyword.replace(' ', '').replace('[', '_').replace(']', '_')
    return keyword

def get_start_index(file):
    # Set the starting position for reading the header
    idx_start = 71 #67
    file.seek(idx_start, os.SEEK_SET)

    no_prop = True

    # Search for the "PROP" keyword in the header
    while file:
        keyword = read_keyword(file)

        if keyword == "PROP":
            idx_start = file.tell() - 8
            no_prop = False
            break
        elif keyword == "AXES":
            file.seek(72, os.SEEK_CUR)
        elif keyword == "GRID":
            file.seek(8, os.SEEK_CUR)
        elif keyword == "GRID_0":
            file.seek(44, os.SEEK_CUR)

    return -1 if no_prop else idx_start

def read_gsg(filename):
    try:
        with open(filename, "rb") as file:
            idx_start = get_start_index(file)
            print('start',idx_start)
            out = GsgOutput()

            if idx_start < 0:
                print("No properties in the file!")
                return out

            file.seek(idx_start, os.SEEK_SET)

            while True:
                keyword = read_keyword(file)
                print('A', keyword)
                if keyword == "CASE_PROPS":
                    print("All properties have been read successfully!")
                    break

                file.seek(8, os.SEEK_CUR)

                keyword = read_keyword(file)
                print('B', keyword)
                file.seek(4, os.SEEK_CUR)

                type_data = struct.unpack("q", file.read(8))[0]
                array_size = struct.unpack("i", file.read(4))[0]
                print('C', type_data, array_size)

                data = GsgData()

                if type_data == 0:
                    data.int_data = list(struct.unpack(f"{array_size}i", file.read(array_size * 4)))
                elif type_data == 1:
                    data.float_data = list(struct.unpack(f"{array_size}f", file.read(array_size * 4)))
                elif type_data == 2:
                    data.double_data = list(struct.unpack(f"{array_size}d", file.read(array_size * 8)))
                else:
                    print("No valid type!")
                    continue

                out.headers.append(keyword)
                out.data_vector.append(data)
            return out

    except FileNotFoundError:
        raise RuntimeError("Error: File does not exist or cannot be opened.")

def write_gsg(filename, start_indices, types, data_vectors):
    with open(filename, "r+b") as file:
        for idx_start, type_data, data in zip(start_indices, types, data_vectors):
            file.seek(idx_start, os.SEEK_SET)

            if type_data == 0:
                file.write(struct.pack(f"{len(data.int_data)}i", *data.int_data))
            elif type_data == 1:
                file.write(struct.pack(f"{len(data.float_data)}f", *data.float_data))
            elif type_data == 2:
                file.write(struct.pack(f"{len(data.double_data)}d", *data.double_data))
            else:
                print("No valid type!")

    print("Data has been written successfully!")


read_gsg(gsg1)


In [ ]:
from collections import namedtuple
from functools import partial
from itertools import batched
from math import prod
from re import finditer, compile as re_compile
from struct import pack, unpack
from os import SEEK_CUR, SEEK_END
from attr import dataclass
import numpy as np

from IORlib.utils import flatten, take

class GSG_File:
    def __init__(self, file_path, write=False):
        self.file_path = Path(file_path)
        self.mode = 'rb'
        if write:
            self.mode = 'wb+'
        self.startpos = 0
        self.version = ''

    def __str__(self):
        return f'{self.file_path}, version: {self.version}' 

    def __enter__(self):
        self.file_obj = open(self.file_path, self.mode)
        if self.mode == 'rb':
            # Jump to postion after 'PetrelForIx'
            self.skip(43) 
            # Read size of version tag
            n = unpack('i', self.read(4))[0] 
            self.version = self.read(n).decode('utf-8')
            self.skip(16)
            self.startpos = self.file_obj.tell()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.file_obj:
            self.file_obj.close()   

    def read(self, *args, **kwargs):
        return self.file_obj.read(*args, **kwargs)

    def keyword_size(self):
        # Read the size of the keyword
        return unpack('<i', self.file_obj.read(4))[0]
    
    def read_keyword(self):
        return self.read(self.keyword_size()).decode('utf-8')

    def write_keyword(self, keyword:str, format:str, header:tuple):
        self.write(pack(f'<i{len(keyword)}s'+format, len(keyword), keyword.encode(), *header))

    def keyword_positions(self, *keywords):
        chunk_size = 1024
        pattern = re_compile(rb'(' + rb'|'.join(k.encode() for k in keywords) + rb')')
        current_pos = self.current_pos()
        self.goto(0)
        try:
            for i,chunk in enumerate(iter(partial(self.read, chunk_size), b'')):
                for match in pattern.finditer(chunk):
                    pos = match.end() + i*chunk_size
                    yield (match.group().decode('utf-8'), pos+4)
        finally:
            self.goto(current_pos)

    def skip(self, pos):
        self.file_obj.seek(pos, SEEK_CUR)

    def end(self, pos):
        self.file_obj.seek(pos, SEEK_END)

    def goto(self, pos):
        self.file_obj.seek(pos)

    def current_pos(self):
        return self.file_obj.tell()

    def size(self):
        return self.file_path.stat().st_size

    def write(self, *args, **kwargs):
        return self.file_obj.write(*args, **kwargs)

    def header(self, version='2023'):
        nver = len(version)
        header = (1, 1, 11, b'PetrelForIx', nver, version.encode(), 0, 0, 0, 0)
        return b'GSG000_b\r\n1\n2\r34\x01\x02\x03\x04' + pack(f'<3i11si{nver}s4i', *header)

    def copy(self, pos1, pos2):
        self.goto(pos1)
        return self.read(pos2-pos1)

    def read_pillars(self, head):
        if head[-3] != 0:
            # Complex grid not yet handled
            return None    
        # Regular grid
        nfloat = 4 + head[3]
        for _ in range(head[2]):
            yield unpack(f'<i{nfloat}f', self.read(4*(1+nfloat)))
            #return unpack('<'+n*f'i{nfloat}f', self.read(4*n*(1+nfloat)))



class GSG_Data():
    bytesize = {'i':4, 'f':4, 'd':8}
    type2format = {0:'i', 1:'f', 2:'d'}
    format2type = {v:k for k,v in type2format.items()}

    def __init__(self, file_path):
        self.file_path = Path(file_path).with_suffix('.GSG')
        self.header_size = 12
        with GSG_File(self.file_path) as file:
            # Process file up to data-array
            # Skip int + 'PROP' + 2int
            file.skip(16)
            self.varname = file.read_keyword()
            file.skip(4)
            # Read data header
            datatype, self.arraysize = unpack('<qi', file.read(self.header_size))
            self.dataformat = self.type2format[datatype]
            self.datasize = self.arraysize * self.bytesize[self.dataformat]
            self.start_data = file.current_pos()
            self.end_data = self.start_data + self.datasize

    def read(self):
        with GSG_File(self.file_path) as file:
            #print(file.current_pos())
            file.goto(self.start_data)
            return unpack(f'<{self.arraysize}{self.dataformat}', file.read(self.datasize))

    def write(self, data, filename, fmt='f'):
        filename = Path(filename).with_suffix('.GSG')
        # Copy head before data and tail after data
        with GSG_File(self.file_path) as file:    
            head = file.copy(0, self.start_data - self.header_size)
            tail = bytearray(file.copy(self.end_data, file.size()))
        # Update the two last uints of the file to reflect the size
        datasize = len(data)*self.bytesize[fmt]
        new_size = len(head) + self.header_size + datasize + len(tail)
        tail[-16:] = pack('<2q', new_size-119, new_size-71)
        # Write to the new file
        with open(filename, 'wb') as file:    
            file.write(head)
            file.write(pack('<qi', self.format2type[fmt], len(data)))
            file.write(pack(f'<{len(data)}{fmt}', *data))
            file.write(tail)
        print(f'   Wrote {filename}')
            

class GSG_Grid():
    Data = namedtuple('grid', 'name dim areal pillars')

    def __init__(self, file_path):
        self.file_path = Path(file_path).with_suffix('.GSG')
        with GSG_File(self.file_path) as file:
            # Skip int + 'AXES' + int + 12char + 2int = 32
            file.skip(32)
            self.axes = unpack('<6f', file.read(24))
            # Skip int + 'GRID' + 2int = 16
            file.skip(16)
            # Grid name and dimensions
            self.varname = file.read_keyword()
            file.skip(4)
            self.start_dim = file.current_pos()
            self.dim = unpack('<3i', file.read(12))
            for _ in range(3):
                n = unpack('<i', file.read(4))[0]
                file.skip(n*4)
            self.start_areal = file.current_pos() + 4

    def read(self):
        with GSG_File(self.file_path) as file:
            file.goto(self.start_areal)
            # Areal
            kw = file.read_keyword()
            if kw != 'AREAL':
                raise ValueError("'AREAL' keyword not found or in wrong position")
            file.skip(12)
            n = unpack('<2i', file.read(8))
            size = 6*n[0]
            data = unpack(f'<{size}i', file.read(4*size))
            areal = list(batched(data, 6))
            # Pillars
            kw = file.read_keyword()
            if kw != 'PILLARS':
                raise ValueError("'PILLARS' keyword not found or in wrong position")
            num_pillars, nfloats, grid_type = unpack('<8x2i40xi8x', file.read(68))
            if grid_type != 0:
                raise NotImplementedError('GSG_Axis does not yet handle complex grids')
            size_pillar = nfloats + 5
            format = f'<i{size_pillar-1}f'
            bin_data = (file.read(4*size_pillar) for _ in range(num_pillars))
            pillars = [unpack(format, data) for data in bin_data]
            return self.Data(self.varname, self.dim, areal, pillars)


    def write(self, data:Data, filename):
        filename = Path(filename).with_suffix('.GSG')
        with GSG_File(self.file_path) as infile, GSG_File(filename, write=True) as outfile:
            # Dimension
            outfile.write(infile.copy(0, self.start_dim))
            # This header is only valid for regular grids
            header = (*data.dim, 1, b'scor', 1, b'sp  ', 1, b'vp  ', 0)
            outfile.write(pack('<4i4si4si4si', *header))
            # Areal
            npillars = (data.dim[0] + 1) * (data.dim[1] + 1)
            header = (0, -1, 0, len(data.areal), npillars-len(data.areal))
            outfile.write(pack('<i5s5i', 5, b'AREAL', *header))
            areal = list(flatten(data.areal))
            outfile.write(pack(f'<{len(areal)}i', *areal))
            # Pillars
            header = (0, 0, npillars, len(data.pillars[0])-5, 0, 0, -1, 0, 4, 4, 0, 1, 2, 3, 0, 0, 0)
            outfile.write(pack('<i7s17i', 7, b'PILLARS', *header))
            for pill in data.pillars:
                outfile.write(pack(f'<i{len(pill)-1}f', *pill))            
            outfile.write_keyword('PROP', '2i', (1,0))
            ncells = prod(data.dim)
            outfile.write_keyword('DEFINED_CELLS', '4s6i', (b'c   ', 0, 0, ncells, 1, ncells, 1))
            outfile.write_keyword('CASE_PROPS', 'i4si8si', (0, b's   ', 1, b'c   s   ', 0))
            outfile.write_keyword('DEFINED_CELLS', '4s3i', (b'g   ', 1, 0, 0))
            outfile.write_keyword('INDEX', '2i', (0, 6))
            keys = ('AXES', 'GRID', 'AREAL', 'PILLARS', 'PROP', 'CASE_PROPS')
            for key, pos in take(len(keys), outfile.keyword_positions(*keys)):
                ival = 1 if key=='PROP' else 0
                #print(key, ival, pos)
                outfile.write_keyword(key, 'iq', (ival, pos))
                #break
            #print('##########', outfile.current_pos())
            #outfile.write(pack('<q', ))


    # def read_complex_pillars(self, head):
    #     size = {1:head[3], 2:6, 3:head[3]}
    #     format = {1:f'<{head[3]}f', 2:'<4i2f', 3:f'<{head[3]}f'}
    #     c = 0
    #     n = unpack('<i', self.read(4))[0]
    #     #print(c, n)
    #     while n != 4:
    #         if n == 3:
    #             print(c, n)
    #         yield (n, *unpack(f'<{size[n]}f', self.read(4*size[n])))
    #         #yield (n, *unpack(format[n], self.read(4*size[n])))
    #         n = unpack('<i', self.read(4))[0]
    #         if n > 1000:
    #             #print(c, n)
    #             self.skip(-4)
    #             vals = []
    #             while True:
    #                 v = unpack('<f', self.read(4))[0]
    #                 if 0<v<1e-20:
    #                     self.skip(-4)
    #                     n = unpack('<i', self.read(4))[0]
    #                     break 
    #                 vals.append(v)
    #             if n==3:
    #                 print('###', c, n, vals)
    #         c += 1
    #     self.skip(-4)

    # def seek(self, pos):
    #     self.file_obj.seek(pos)



def read_prop(gsg, **kwargs):
    with GSG_File(gsg, **kwargs) as file:
        print(file)
        # PROP
        kw = file.read_keyword()
        print(kw, unpack('<2i', file.read(8)))
        # POROS
        kw = file.read_keyword()
        #file.skip(4)
        a = file.read(4)
        var, data = file.read_data()
        print(kw, a, len(data))
        print(data[:10])
        print(data[-10:])
        # CASE_PROPS
        kw = file.read_keyword()
        print(kw, unpack('<7i', file.read(28)))
        # POROSITY
        kw = file.read_keyword()
        print(kw, unpack('<3i', file.read(12)))
        # INDEX
        kw = file.read_keyword()
        print(kw, unpack('<2i', file.read(8)))
        # PROP
        kw = file.read_keyword()
        print(kw, unpack('<3i', file.read(12)))
        # CASE_PROPS
        kw = file.read_keyword()
        end = unpack('<5i', file.read(20))
        print(kw, end)
        endpos = file.file_obj.tell()
        print(endpos-end[-2], endpos-end[1])

def read_axes(gsg):
    with GSG_File(gsg) as file:
        print(file)
        # AXES
        print(file.current_pos())
        kw = file.read_keyword()
        print(kw, unpack('<i12s2i6f', file.read(48)))
        # GRID
        print(file.current_pos())
        kw = file.read_keyword()
        print(kw, unpack('<2i', file.read(8)))
        # name
        kw = file.read_keyword()
        print(kw, unpack('<4i', file.read(16)), end='')
        for _ in range(3):
            n = unpack('<i', file.read(4))[0]
            print(unpack(f'{n*4}s', file.read(n*4)), end='')
        print(unpack('<i', file.read(4)))
        # AREAL
        kw = file.read_keyword()
        print(file.current_pos()+4)
        n = unpack('<5i', file.read(20))
        size = 6*n[-2]
        data = unpack(f'<{size}i', file.read(4*size))
        areal = list(batched(data,6))
        print(kw, n, areal[:2], '...', areal[-2:])
        # PILLARS
        kw = file.read_keyword()
        print(file.current_pos()+4)
        n = unpack('<17i', file.read(68))
        print(kw, n)
        pillars = list(file.read_pillars(n))
        #print(pillars[0])
        if not pillars:
            return None
        print(len(pillars), pillars[:2], '...', pillars[-2:])
        # PROP
        kw = file.read_keyword()
        print(kw, unpack('<2i', file.read(8)))
        # DEFINED_CELLS
        kw = file.read_keyword()
        print(kw, unpack('<4s6i', file.read(28)))
        # CASE_PROPS
        kw = file.read_keyword()
        print(kw, unpack('<i4si8si', file.read(24)))
        # DEFINED_CELLS
        kw = file.read_keyword()
        print(kw, unpack('<4s3i', file.read(16)))
        # INDEX
        kw = file.read_keyword()
        print(kw, unpack('<2i', file.read(8)))
        # AXES
        kw = file.read_keyword()
        print(kw, unpack('<3i', file.read(12)))
        # GRID
        kw = file.read_keyword()
        print(kw, unpack('<3i', file.read(12)))
        # AREAL
        kw = file.read_keyword()
        print(kw, unpack('<3i', file.read(12)))
        # PILLARS
        kw = file.read_keyword()
        print(kw, unpack('<3i', file.read(12)))
        # PROP
        kw = file.read_keyword()
        print(kw, unpack('<3i', file.read(12)))
        # CASE_PROPS
        kw = file.read_keyword()
        print(kw, unpack('<5i', file.read(20)))
        print('size:', file.size())

#gsg = GSG_Data(gsg1)
#gsg.read()
#GSG_Data(gsg1).write(list(range(10)), filename='TEST')
#GSG_Data('TEST.GSG').read()
#GSG_Data(gsg8).fix_end()
read_prop(gsg1)


In [ ]:
def f(*a):
    print(a)

A = (('A',0),('B',1))
f(*dict(A).keys())

In [ ]:
with GSG_File(gsg7) as file:
    keys = ('AXES', 'GRID', 'AREAL', 'PILLARS', 'PROP', 'CASE_PROPS')
    pos_key = take(len(keys), file.keyword_positions(*keys))
pos_key

In [ ]:
from functools import partial
from re import finditer

def find():
    with open(gsg3, 'rb') as file:
        chunk_size = 1024
        pos = 0
        for i,chunk in enumerate(iter(partial(file.read, chunk_size), b'')):
            for match in finditer(rb'(AXES|GRID|AREAL|PILLARS|PROP)', chunk):
                pos = match.end() + i*chunk_size
                print(match.group(), pos+4)
                if match.group() == b'PROP':
                    return
find()
#print(match, pos)

In [ ]:
from IORlib.ECL import EGRID_file

root = 'cases/L18_FWD/L18_FWD'
#list(EGRID_file(root).blockdata())
for block in EGRID_file(root1).blocks():
    print(block)
    print(block.data())

In [ ]:
300 - 0.11015419661998749